# Stage 3 — Full E5 fine-tuning with in-batch negatives

Run the approved smoke test first, then the single-seed pilot. This notebook reads only the fixed Stage 1 train and validation splits; it never reads or evaluates SciFact test data. All inputs, checkpoints, metrics, and re-embedded corpus vectors persist under `MyDrive/RAFT`.

In [ ]:
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from zipfile import ZipFile

MY_DRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = MY_DRIVE / 'RAFT'
BOOTSTRAP_ARCHIVE = MY_DRIVE / 'stage3_drive_bootstrap.zip'
READY_MARKERS = (
    DRIVE_ROOT / 'stage1_scifact' / 'processed' / 'corpus.jsonl',
    DRIVE_ROOT / 'stage2_baselines' / 'results' / 'stage2_baselines.csv',
    DRIVE_ROOT / 'stage3_inbatch' / 'source_snapshot' / 'src' / 'train.py',
)
if not all(path.is_file() for path in READY_MARKERS):
    assert BOOTSTRAP_ARCHIVE.is_file(), f'Missing Drive bootstrap archive: {BOOTSTRAP_ARCHIVE}'
    with ZipFile(BOOTSTRAP_ARCHIVE) as archive:
        archive.extractall(MY_DRIVE)
assert all(path.is_file() for path in READY_MARKERS)
print(f'Restartable RAFT artifacts ready under {DRIVE_ROOT}')

In [ ]:
import json
import os
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
DRIVE_SOURCE = DRIVE_ROOT / 'stage3_inbatch' / 'source_snapshot'
REPO_DIR = DRIVE_SOURCE if (DRIVE_SOURCE / 'src' / 'train.py').is_file() else Path('/content/RAFT')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/zaidamir-hash/raft-scifact.git', str(REPO_DIR)], check=True)

DATA_DIR = DRIVE_ROOT / 'stage1_scifact' / 'processed'
BASELINE_RESULTS = DRIVE_ROOT / 'stage2_baselines' / 'results' / 'stage2_baselines.csv'
CONFIG = REPO_DIR / 'configs' / 'stage3_inbatch_pilot_v2.json'
SCRIPT = REPO_DIR / 'src' / 'train.py'
SMOKE_DIR = DRIVE_ROOT / 'stage3_inbatch' / 'smoke_seed42'
PILOT_DIR = DRIVE_ROOT / 'stage3_inbatch' / 'pilot_seed42_lr5e6_eval5'
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'huggingface_cache')

for required in (
    DATA_DIR / 'corpus.jsonl',
    DATA_DIR / 'queries_train.jsonl',
    DATA_DIR / 'qrels_train.tsv',
    DATA_DIR / 'queries_validation.jsonl',
    DATA_DIR / 'qrels_validation.tsv',
    BASELINE_RESULTS, CONFIG, SCRIPT,
):
    assert required.is_file(), f'Missing restartable Stage 3 input: {required}'
print(f'Using restartable source: {REPO_DIR}')
print(f'Using Drive data: {DATA_DIR}')

## 1. Smoke test

This must pass before the pilot cell is run. It executes exactly three optimizer steps on 96 deterministic training pairs and performs no validation or test evaluation.

In [ ]:
smoke_command = [
    'python', str(SCRIPT),
    '--mode', 'smoke',
    '--config', str(CONFIG),
    '--data-dir', str(DATA_DIR),
    '--baseline-results', str(BASELINE_RESULTS),
    '--output-dir', str(SMOKE_DIR),
]
subprocess.run(smoke_command, check=True)
smoke_result = json.loads((SMOKE_DIR / 'results' / 'stage3_smoke_result.json').read_text())
assert smoke_result['passed'] is True
assert smoke_result['global_steps_completed'] == 3
assert smoke_result['test_split_loaded'] is False
print('SMOKE TEST PASSED — pilot may now run.')

## 2. Approved revised single-seed pilot

The original 2e-5/epoch-eval pilot degraded all validation metrics. This user-approved v2 lowers the learning rate to 5e-6 and evaluates/saves every 5 steps. It reuses the already-passed smoke result because the loss/training path is unchanged and smoke mode performs no evaluation. Checkpoint selection and early stopping use validation nDCG@10 only.

In [ ]:
smoke_result = json.loads((SMOKE_DIR / 'results' / 'stage3_smoke_result.json').read_text())
assert smoke_result['passed'] is True and smoke_result['global_steps_completed'] == 3
assert smoke_result['test_split_loaded'] is False
print('REUSING PASSED SMOKE TEST — core training code and loss are unchanged.')
pilot_command = [
    'python', str(SCRIPT),
    '--mode', 'pilot',
    '--config', str(CONFIG),
    '--data-dir', str(DATA_DIR),
    '--baseline-results', str(BASELINE_RESULTS),
    '--output-dir', str(PILOT_DIR),
]
subprocess.run(pilot_command, check=True)

In [ ]:
import pandas as pd
pilot_result = json.loads((PILOT_DIR / 'results' / 'stage3_pilot_result.json').read_text())
assert pilot_result['test_split_loaded'] is False
display(pd.read_csv(PILOT_DIR / 'results' / 'stage3_pilot_validation_comparison.csv'))
display(pd.DataFrame([pilot_result['loss_summary']]))
print('Best checkpoint:', pilot_result['saved_best_checkpoint'])
print('Corpus cache:', PILOT_DIR / 'embeddings')